## Figures

In [ ]:
import numpy as np
import pandas as pd

file_path = r"E:\Data\Ralstonia\Pycharm\graduation2\new_analysis\association\mapping_with_metabolism_species_reclassified_broad_source.csv"
raw = pd.read_csv(file_path, encoding="gb18030")

# broad_source 中的 dicotyledon / monocotyledon 用于植物分层；
# Soil/rhizosphere 和 Water 单独作为 isolation source 面板。
plant_data = raw.loc[
    raw["broad_source"].isin(["monocotyledon", "dicotyledon"])
    & raw["host_norm"].notna()
    & raw["family"].notna()
].copy()

trait_cols = {
    "Disaccharide utilization": "disaccharide utilization",
    "Hexitol utilization": "hexitol utilization",
}

source_data = raw.loc[
    raw["broad_source"].isin(["soil/rhizosphere", "water"])
].copy()

level_cols = {
    "broad_source": "broad_source",
    "species": "host_norm",
    "family": "family",
    "cotyledon": "broad_source",
}

data_by_level = {
    "broad_source": source_data,
    "species": plant_data,
    "family": plant_data,
    "cotyledon": plant_data,
}

min_n_by_level = {
    "broad_source": 1,
    "species": 10,
    "family": 10,
    "cotyledon": 1,
}


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half_width = z * np.sqrt(
        p * (1 - p) / n + z**2 / (4 * n**2)
    ) / denom

    return max(0, center - half_width), min(1, center + half_width)


rows = []

for level, group_col in level_cols.items():
    level_data = data_by_level[level]
    min_n = min_n_by_level[level]

    for label, g in level_data.groupby(group_col, dropna=True):
        n = len(g)

        if n < min_n:
            continue

        for trait_name, trait_col in trait_cols.items():
            positive = int(g[trait_col].sum())
            negative = n - positive
            prevalence = positive / n
            ci_low, ci_high = wilson_ci(positive, n)

            rows.append({
                "level": level,
                "label": label,
                "trait": trait_name,
                "n": n,
                "positive": positive,
                "negative": negative,
                "prevalence": prevalence,
                "ci_low": ci_low,
                "ci_high": ci_high,
            })

prevalence_ci_results = pd.DataFrame(rows)

prevalence_ci_results = prevalence_ci_results.sort_values(
    ["level", "n", "label", "trait"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

print("plant samples:", len(plant_data))
print("environmental samples (soil + water):", len(source_data))
print("result rows:", len(prevalence_ci_results))
print(prevalence_ci_results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['legend.fontsize'] = 14
# =========================================================
# 1. 读取 phyloglm 结果
# =========================================================
phylo_path = r"E:\Data\Ralstonia\Pycharm\graduation2\new_analysis\association\final_phyloglm_primary_newdata\phyloglm_association_models.tsv"
phylo = pd.read_csv(phylo_path, sep="\t")

num_cols = [
    "phyloglm_OR",
    "phyloglm_CI_low",
    "phyloglm_CI_high",
    "phyloglm_P",
    "fdr",
    "focal_prevalence",
]
for c in num_cols:
    if c in phylo.columns:
        phylo[c] = pd.to_numeric(phylo[c], errors="coerce")

phylo["separation_bool"] = (
    phylo["separation"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

trait_map = {
    "disaccharide_use": "Disaccharide utilization",
    "hexose_alcohol_use": "Hexitol utilization",
}

# =========================================================
# 2. 整理 phyloglm 气泡图数据
# =========================================================
assoc_rows = []

for _, r in phylo.iterrows():
    trait = trait_map.get(r["metabolic_trait"])
    if trait is None:
        continue

    ecological_level = str(r["ecological_level"])
    contrast = r["contrast"]

    # species / family
    if ecological_level.startswith("species_"):
        level = "species"
        label = contrast

    elif ecological_level.startswith("family_"):
        level = "family"
        label = contrast

    # broad isolation source: report the two environmental focal groups
    elif ecological_level == "broad_source":
        level = "broad_source"
        test_id = str(r["test_id"])
        if test_id.startswith("src_soil_rhizosphere_"):
            label = "soil/rhizosphere"
        elif test_id.startswith("src_water_"):
            label = "water"
        else:
            continue

    # cotyledon: phyloglm 只有 host_monocot 这一项
    # monocotyledon 使用原始 OR
    # dicotyledon 使用倒数 OR，表示相对 monocot 的方向
    elif ecological_level == "monocot_eudicot":
        level = "cotyledon"

        if pd.notna(r["phyloglm_OR"]):
            or_mono = r["phyloglm_OR"]

            assoc_rows.append({
                "level": level,
                "label": "monocotyledon",
                "trait": trait,
                "phyloglm_OR": or_mono,
                "fdr": r["fdr"],
                "separation": False,
            })

            assoc_rows.append({
                "level": level,
                "label": "dicotyledon",
                "trait": trait,
                "phyloglm_OR": 1 / or_mono,
                "fdr": r["fdr"],
                "separation": False,
            })

        continue

    else:
        continue

    assoc_rows.append({
        "level": level,
        "label": label,
        "trait": trait,
        "phyloglm_OR": r["phyloglm_OR"],
        "fdr": r["fdr"],
        "separation": bool(r["separation_bool"]),
    })

assoc_results = pd.DataFrame(assoc_rows)

assoc_results["log2_OR"] = np.log2(assoc_results["phyloglm_OR"])
assoc_results["abs_log2_OR"] = assoc_results["log2_OR"].abs()
assoc_results["direction"] = np.where(
    assoc_results["phyloglm_OR"] > 1,
    "enriched",
    "depleted",
)
assoc_results.loc[assoc_results["separation"], "direction"] = "enriched"
assoc_results["fdr_sig"] = assoc_results["fdr"] < 0.05

# =========================================================
# 3. 画图参数
# =========================================================
prev = prevalence_ci_results.copy()

trait_order = [
    "Disaccharide utilization",
    "Hexitol utilization",
]

level_order = [
    ("broad_source", "A."),
    ("cotyledon", "B."),
    ("family", "C."),
    ("species", "D."),
]

forest_colors = {
    "Disaccharide utilization": "#E08A5B",
    "Hexitol utilization": "#5BA6C8",
}

bubble_colors = {
    "enriched": "#E08A5B",   # 红色
    "depleted": "#5BA6C8",   # 蓝色
}

label_display = {
    "monocotyledon": "monocotyledon",
    "dicotyledon": "dicotyledon",
    "soil/rhizosphere": "Soil/rhizosphere",
    "water": "Water",
}

# 气泡大小：用 |log2(OR)|
normal_max_abs = assoc_results.loc[
    assoc_results["abs_log2_OR"].notna() & ~assoc_results["separation"],
    "abs_log2_OR"
].max()

if pd.isna(normal_max_abs) or normal_max_abs == 0:
    normal_max_abs = 1

min_size = 70
max_size = 450

def bubble_size(abs_log2_or, separation=False):
    if separation:
        return max_size
    if pd.isna(abs_log2_or):
        return 0
    return min_size + (abs_log2_or / normal_max_abs) * (max_size - min_size)


# =========================================================
# 4. 设置每个 level 的行顺序
# =========================================================
def get_label_order(level):
    d = prev.loc[
        (prev["level"] == level)
        & (prev["trait"] == "Disaccharide utilization")
    ].copy()

    available = set(d["label"])

    if level == "broad_source":
        order = ["soil/rhizosphere", "water"]
        return [x for x in order if x in available]

    if level == "cotyledon":
        order = ["monocotyledon", "dicotyledon"]
        return [x for x in order if x in available]

    if level == "family":
        order = [
            "Musaceae",
            "Zingiberaceae",
            "Araceae",
            "Solanaceae",
            "Fabaceae",
            "Rosaceae",
            "Cucurbitaceae",
        ]
        return [x for x in order if x in available]

    if level == "species":
        order = [
            "Curcuma spp.",
            "Musa spp.",
            "Zingiber officinale",
            "Capsicum annuum",
            "Nicotiana tabacum",
            "Rosa spp.",
            "Solanum dulcamara",
            "Solanum lycopersicum",
            "Solanum melongena",
            "Solanum tuberosum",
            "Solanum virginianum",
        ]
        return [x for x in order if x in available]

    return (
        d.sort_values(["n", "label"], ascending=[False, True])
        ["label"]
        .tolist()
    )


# =========================================================
# 5. 正式画图
# =========================================================
height_ratios = [len(get_label_order(level)) for level, _ in level_order]

fig = plt.figure(figsize=(15, 12.5))
gs = fig.add_gridspec(
    nrows=4,
    ncols=4,
    width_ratios=[1.25, 1.25, 1.05, 0.1],
    height_ratios=height_ratios,
    wspace=0.16,
    hspace=0.4,
)

for row_i, (level, panel_title) in enumerate(level_order):
    labels = get_label_order(level)
    y_pos = np.arange(len(labels))

    ax_dis = fig.add_subplot(gs[row_i, 0])
    ax_hex = fig.add_subplot(gs[row_i, 1], sharey=ax_dis)
    ax_bub = fig.add_subplot(gs[row_i, 2], sharey=ax_dis)
    ax_n = fig.add_subplot(gs[row_i, 3], sharey=ax_dis)

    forest_axes = {
        "Disaccharide utilization": ax_dis,
        "Hexitol utilization": ax_hex,
    }

    # -----------------------------
    # 左边 prevalence forest plot
    # -----------------------------
    for trait, ax in forest_axes.items():
        d = (
            prev.loc[
                (prev["level"] == level)
                & (prev["trait"] == trait)
            ]
            .set_index("label")
            .reindex(labels)
            .reset_index()
        )

        x = d["prevalence"].astype(float)
        xerr_left = (x - d["ci_low"].astype(float)).clip(lower=0)
        xerr_right = (d["ci_high"].astype(float) - x).clip(lower=0)

        ax.errorbar(
            x,
            y_pos,
            xerr=np.vstack([xerr_left, xerr_right]),
            fmt="o",
            color=forest_colors[trait],
            ecolor=forest_colors[trait],
            elinewidth=1.6,
            capsize=2.5,
            markersize=5,
            zorder=3,
        )

        ax.set_xlim(-0.03, 1.03)
        ax.set_xticks(np.linspace(0, 1, 6))
        ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
        ax.grid(axis="y", color="#eeeeee", linewidth=0.8)
        ax.set_ylim(-0.5, len(labels) - 0.5)
        ax.invert_yaxis()

        if row_i == 0:
            ax.set_title(trait, fontsize=12, fontweight="bold",
                         color=forest_colors[trait])

        if row_i == len(level_order) - 1:
            ax.set_xlabel("Prevalence")

    ax_dis.set_yticks(y_pos)
    ax_dis.set_yticklabels(
        [label_display.get(x, x) for x in labels]
    )
    ax_dis.set_title(panel_title, loc="left", fontweight="bold")

    plt.setp(ax_hex.get_yticklabels(), visible=False)
    ax_hex.tick_params(axis="y", length=0)

    # -----------------------------
    # 中间 n
    # -----------------------------
    n_data = (
        prev.loc[
            (prev["level"] == level)
            & (prev["trait"] == "Disaccharide utilization")
        ]
        .set_index("label")
        .reindex(labels)
        .reset_index()
    )

    ax_n.set_xlim(0, 1)
    ax_n.set_ylim(-0.5, len(labels) - 0.5)
    ax_n.invert_yaxis()
    ax_n.axis("off")

    # if row_i == 0:
    #     ax_n.set_title("Sample size", fontsize=11, fontweight="bold")

    for yi, n in zip(y_pos, n_data["n"]):
        ax_n.text(
            -1.3,
            yi,
            f"n={int(n)}",
            va="center",
            fontsize=15,
            ha="left",
            color="#444444",
            clip_on=False
        )

    # -----------------------------
    # 右边 phyloglm bubble plot
    # -----------------------------
    ax_bub.set_xlim(-0.55, 1.55)
    ax_bub.set_ylim(-0.5, len(labels) - 0.5)
    ax_bub.invert_yaxis()
    ax_bub.set_xticks([0, 1])
    ax_bub.set_xticklabels(["Disaccharide", "Hexitol"])
    ax_bub.set_yticks(y_pos)
    ax_bub.tick_params(axis="y", length=0)
    plt.setp(ax_bub.get_yticklabels(), visible=False)

    ax_bub.grid(axis="y", color="#eeeeee", linewidth=0.8)
    ax_bub.grid(axis="x", color="#eeeeee", linewidth=0.8)

    if row_i == 0:
        ax_bub.set_title("Phyloglm association", fontweight="bold")

    for trait_i, trait in enumerate(trait_order):
        ad = assoc_results.loc[
            (assoc_results["level"] == level)
            & (assoc_results["trait"] == trait)
        ].copy()

        for yi, label in zip(y_pos, labels):
            r = ad.loc[ad["label"] == label]

            if r.empty:
                continue

            r = r.iloc[0]

            if r["separation"]:
                ax_bub.scatter(
                    trait_i,
                    yi,
                    s=max_size,
                    marker="o",
                    facecolors="none",
                    color=bubble_colors["enriched"],
                    edgecolor=bubble_colors["enriched"],
                    linewidth=1.1,
                    zorder=4,
                )
                continue

            if pd.isna(r["phyloglm_OR"]):
                continue

            color = bubble_colors["enriched"] if r["phyloglm_OR"] > 1 else bubble_colors["depleted"]
            size = bubble_size(r["abs_log2_OR"], separation=False)

            if r["fdr_sig"]:
                edgecolor = "#4A4A4A"
                linewidth = 1.4
            else:
                edgecolor = "white"
                linewidth = 0.35

            ax_bub.scatter(
                trait_i,
                yi,
                s=size,
                marker="o",
                color=color,
                alpha=0.95,
                edgecolor=edgecolor,
                linewidth=linewidth,
                zorder=4,
            )

    for ax in [ax_dis, ax_hex, ax_bub]:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# =========================================================
# 6. 图例
# =========================================================

# -----------------------------
# 6.1 颜色 / 显著性 / complete separation 图例
# -----------------------------
legend_handles_main = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        markerfacecolor=bubble_colors["enriched"],
        markeredgecolor="none",
        markersize=16,
        label="OR > 1 enriched",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        markerfacecolor=bubble_colors["depleted"],
        markeredgecolor="none",
        markersize=16,
        label="OR < 1 depleted",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        markerfacecolor="white",
        markeredgecolor="#4A4A4A",
        markeredgewidth=1.4,
        markersize=16,
        label="FDR q < 0.05",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor=bubble_colors["enriched"],
        markeredgewidth=1.2,
        markersize=16,
        label="Complete separation",
    ),
]

legend1 = fig.legend(
    handles=legend_handles_main,
    loc="upper right",
    bbox_to_anchor=(1.08, 0.955),
    frameon=False,
    handletextpad=0.8,
    labelspacing=0.6,
)

fig.add_artist(legend1)


# -----------------------------
# 6.2 气泡大小图例：|log2(OR)|
# -----------------------------
size_values = [
    0,
    normal_max_abs / 2,
    normal_max_abs,
]

size_labels = [
    "0",
    f"{normal_max_abs / 2:.1f}",
    f"{normal_max_abs:.1f}",
]

size_handles = [
    plt.scatter(
        [],
        [],
        s=bubble_size(v, separation=False),
        facecolors="none",
        edgecolors="#666666",
        linewidths=1.0,
        label=lab,
    )
    for v, lab in zip(size_values, size_labels)
]

legend2 = fig.legend(
    handles=size_handles,
    title=r"|log$_2$(OR)|",
    fontsize=16,
    title_fontsize=18,
    loc="upper right",
    bbox_to_anchor=(1.01, 0.785),
    frameon=False,
    scatterpoints=1,
    labelspacing=1.2,
    handletextpad=1.2,
)

fig.add_artist(legend2)

plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig(
    r"E:\Data\Ralstonia\article\img\figure7\association3.svg",
    transparent=True,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# Part 1. 读取并整理数据
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

file_path = r"E:\Data\Ralstonia\Pycharm\graduation2\new_analysis\association\template.xlsx"

sig_raw = pd.read_excel(file_path, sheet_name="Sheet13")
or_raw = pd.read_excel(file_path, sheet_name="Sheet14")

# 清理列名
sig_raw.columns = sig_raw.columns.astype(str).str.strip()
or_raw.columns = or_raw.columns.astype(str).str.strip()

# 按热图想展示的列顺序
sig_cols = [
    "phyloglm_all_sig",
    "geography_sig",
    "experimental_only_sig",
    "pcmds_sig",
    "kinship_glmm_direction",
    "phyloglm_same_cotyledon_sig",
    "pagel_vs_all_bootstrap_sig",
    "pagel_within_clade_bootstrap_sig",
]

# 对应到 Sheet14 里的 OR / effect-size 列
# Pagel 没有 OR，所以不写方向，只用颜色表示 bootstrap_P 是否显著
or_col_map = {
    "phyloglm_all_sig": "phyloglm_all_OR",
    "geography_sig": "geography_OR",
    "experimental_only_sig": "experimental_only_OR",
    "pcmds_sig": "pcmds_OR",
    "kinship_glmm_direction": "kinship_glmm_OR",
    "phyloglm_same_cotyledon_sig": "phyloglm_same_cotyledon_OR",
    "pagel_vs_all_bootstrap_sig": None,
    "pagel_within_clade_bootstrap_sig": None,
}

# 合并显著性表和 OR 表
meta_cols = ["level", "host_clade", "label"]

sig_df = sig_raw[meta_cols + sig_cols].copy()
or_cols_needed = [
    c for c in or_col_map.values()
    if c is not None and c in or_raw.columns
]

or_df = or_raw[meta_cols + or_cols_needed].copy()

plot_df = sig_df.merge(
    or_df,
    on=meta_cols,
    how="left",
)

# 把显著性矩阵转成画图用的数值：
# 1 -> 1, 0 -> 0, NA / complete separation -> NaN
heatmap_values = pd.DataFrame(index=plot_df.index)

# 格子里要写的文字：+ / - / enriched
annot_text = pd.DataFrame("", index=plot_df.index, columns=sig_cols)

for col in sig_cols:
    vals = plot_df[col]

    heatmap_values[col] = np.nan

    or_col = or_col_map[col]

    for i, v in vals.items():

        # 先统一处理成字符串，但保留 NA 判断
        if pd.isna(v):
            v_str = "NA"
        else:
            v_str = str(v).strip()

        v_lower = v_str.lower()

        # Kinship-GLMM 本身就是方向列，不参与显著性着色
        if col == "kinship_glmm_direction":
            if v_str in ["+", "-"]:
                annot_text.loc[i, col] = v_str
            else:
                annot_text.loc[i, col] = ""
            heatmap_values.loc[i, col] = np.nan
            continue

        # 1 / 0 转成颜色矩阵
        if v_str in ["1", "1.0"]:
            heatmap_values.loc[i, col] = 1
        elif v_str in ["0", "0.0"]:
            heatmap_values.loc[i, col] = 0

        # complete separation 单独标 enriched，不给颜色
        if v_lower == "complete separation":
            annot_text.loc[i, col] = "enriched"
            heatmap_values.loc[i, col] = np.nan
            continue

        # NA 空白
        if v_str.upper() == "NA" or v_str == "" or v_lower == "nan":
            heatmap_values.loc[i, col] = np.nan
            continue

        # Pagel 没有 OR，不写 +/-
        if or_col is None or or_col not in plot_df.columns:
            continue

        # 只有显著结果才写 +/-
        if heatmap_values.loc[i, col] != 1:
            continue

        or_value = plot_df.loc[i, or_col]

        if pd.isna(or_value):
            continue

        or_str = str(or_value).strip().lower()

        if or_str == "enriched":
            annot_text.loc[i, col] = "+"
        elif or_str == "depleted":
            annot_text.loc[i, col] = "-"
        else:
            try:
                or_float = float(or_value)
                if or_float > 1:
                    annot_text.loc[i, col] = "+"
                elif or_float < 1:
                    annot_text.loc[i, col] = "-"
            except ValueError:
                pass



# 行标签
row_labels = plot_df["label"].astype(str).tolist()

# 为了画图好看，给列名换成短一点的显示名
display_col_names = {
    "phyloglm_all_sig": "phyloglm\nall",
    "geography_sig": "phyloglm\ngeography adjusted",
    "experimental_only_sig": "phyloglm\nexperimental only",
    "pcmds_sig": "PC/MDS",
    "kinship_glmm_direction": "kinship GLMM",
    "phyloglm_same_cotyledon_sig": "phyloglm\nsame cotyledon",
    "pagel_vs_all_bootstrap_sig": "Pagel\nall",
    "pagel_within_clade_bootstrap_sig": "Pagel\nsame cotyledon",
}

x_labels = [display_col_names.get(c, c) for c in sig_cols]

plot_df.head()

In [ ]:
# ============================================================
# Part 2. 分 panel 画显著性热图 + OR方向
# ============================================================

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['legend.fontsize'] = 14

# 颜色设置
color_not_sig = "#D8ECF5"   # 浅蓝：不显著
color_sig = "#457b9d"       # 深蓝：显著
color_na = "#FFFFFF"        # 空白：NA / complete separation

cmap = ListedColormap([color_not_sig, color_sig])
norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)

# ------------------------------------------------------------
# 统一行顺序：monocot 在上，dicot 在下
# ------------------------------------------------------------
plot_df = plot_df.copy()
heatmap_values = heatmap_values.copy()
annot_text = annot_text.copy()

level_rank = {
    "broad source": 0,
    "plant cotyledon": 1,
    "family": 2,
    "species": 3,
}

clade_rank = {
    "NA": 0,
    "nan": 0,
    "monocot": 1,
    "dicot": 2,
}

label_rank = {
    # broad source
    "Soil/rhizosphere": 0,
    "Water": 1,

    # cotyledon
    "Monocotyledon": 0,
    "Dicotyledon": 1,

    # monocot family
    "Musaceae": 0,
    "Zingiberaceae": 1,
    "Araceae": 2,

    # dicot family
    "Solanaceae": 0,
    "Fabaceae": 1,
    "Rosaceae": 2,
    "Cucurbitaceae": 3,

    # monocot species
    "Curcuma spp.": 0,
    "Musa spp.": 1,
    "Zingiber officinale": 2,

    # dicot species
    "Capsicum annuum": 0,
    "Nicotiana tabacum": 1,
    "Rosa spp.": 2,
    "Solanum dulcamara": 3,
    "Solanum lycopersicum": 4,
    "Solanum melongena": 5,
    "Solanum tuberosum": 6,
    "Solanum virginianum": 7,
}

sort_key = pd.DataFrame({
    "level_rank": plot_df["level"].map(level_rank).fillna(99),
    "clade_rank": plot_df["host_clade"].astype(str).map(clade_rank).fillna(99),
    "label_rank": plot_df["label"].map(label_rank).fillna(99),
})

ordered_index = sort_key.sort_values(
    ["level_rank", "clade_rank", "label_rank"],
    kind="stable"
).index

plot_df = plot_df.loc[ordered_index].reset_index(drop=True)
heatmap_values = heatmap_values.loc[ordered_index].reset_index(drop=True)
annot_text = annot_text.loc[ordered_index].reset_index(drop=True)

# 四个 panel，只按 level 分开；不显示 A/B/C/D 标签
panel_order = [
    "broad source",
    "plant cotyledon",
    "family",
    "species",
]

n_cols = len(sig_cols)

# 每个 panel 高度按行数动态分配
height_ratios = []
for level_name in panel_order:
    n = (plot_df["level"].astype(str) == level_name).sum()
    height_ratios.append(max(n, 1))

fig_height = max(7, sum(height_ratios) * 0.36 + 1.6)
fig_width = max(8, n_cols * 0.85 + 2.0)

fig, axes = plt.subplots(
    nrows=len(panel_order),
    ncols=1,
    figsize=(fig_width, fig_height),
    sharex=True,
    gridspec_kw={
        "height_ratios": height_ratios,
        "hspace": 0.2
    }
)

if len(panel_order) == 1:
    axes = [axes]

clade_colors = {
    "monocot": "#e98e3a",
    "dicot": "#1c969c",
    "NA": "#555555",
    "nan": "#555555",
}

for ax, level_name in zip(axes, panel_order):

    idx = plot_df.index[plot_df["level"].astype(str) == level_name].tolist()

    sub_heatmap = heatmap_values.loc[idx, sig_cols].astype(float).values
    sub_annot = annot_text.loc[idx, sig_cols].reset_index(drop=True)
    sub_df = plot_df.loc[idx].reset_index(drop=True)

    n_rows = len(sub_df)

    ax.set_facecolor(color_na)

    masked_matrix = np.ma.masked_invalid(sub_heatmap)

    ax.imshow(
        masked_matrix,
        cmap=cmap,
        norm=norm,
        aspect="auto",
        interpolation="none"
    )

    # 格子边框
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="#E6E6E6", linewidth=0.8)
    ax.tick_params(which="minor", bottom=False, left=False)

    # y 轴标签
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(sub_df["label"].astype(str), fontsize=12)

    for tick, clade in zip(ax.get_yticklabels(), sub_df["host_clade"].astype(str)):
        tick.set_color(clade_colors.get(clade, "#555555"))

    # 写 + / - / enriched
    for i in range(n_rows):
        for j, col in enumerate(sig_cols):

            txt = sub_annot.loc[i, col]

            if txt == "":
                continue

            if txt == "+":
                color = "black"
                fontsize = 16
                fontweight = "normal"
            elif txt == "-":
                color = "black"
                fontsize = 16
                fontweight = "normal"
            elif txt == "enriched":
                color = "black"
                fontsize = 10
                fontweight = "normal"
            else:
                color = "#333333"
                fontsize = 12
                fontweight = "normal"

            ax.text(
                j,
                i,
                txt,
                ha="center",
                va="center",
                fontsize=fontsize,
                color=color,
                fontweight=fontweight
            )

    # 边框处理
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# 只在最后一个 panel 显示 x 轴标签
axes[-1].set_xticks(np.arange(n_cols))
axes[-1].set_xticklabels(
    x_labels,
    rotation=45,
    ha="right",
    va="top",
    fontsize=12
)

# 其他 panel 不显示 x 轴标签、刻度线和底部轴线
for ax in axes[:-1]:
    ax.tick_params(
        axis="x",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False
    )
    ax.spines["bottom"].set_visible(True)

# 图例
legend_handles = [
    Patch(facecolor=color_sig, edgecolor="#E6E6E6", label="Significant"),
    Patch(facecolor=color_not_sig, edgecolor="#E6E6E6", label="Not significant"),
    Patch(facecolor=color_na, edgecolor="#E6E6E6", label="NA / not tested"),
]

axes[0].legend(
    handles=legend_handles,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    fontsize=12
)

fig.suptitle(
    "Sensitivity evidence summary for disaccharide utilization",
    fontsize=13,
    fontweight="bold",
    y=0.98
)

plt.tight_layout(rect=[0, 0, 0.88, 0.96])

# 保存
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
out_pdf = r"E:\Data\Ralstonia\article\img\supplementfigure\association_supplement\sensitivity_evidence_heatmap.pdf"

plt.savefig(out_pdf, transparent=True, bbox_inches="tight")
plt.show()